In [ ]:
%pip install llama-index llama-index-embeddings-cohere llama-index-llms-cohere cohere python-dotenv
%pip install llama-index-vector-stores-pinecone pinecone-client
%pip install gradio

In [2]:
import os, httpx, ssl, urllib3
import gradio as gr
from dotenv import load_dotenv
from pinecone import Pinecone

# LlamaIndex Core & Components
from llama_index.core import (
    Settings, 
    StorageContext, 
    VectorStoreIndex, 
    SimpleDirectoryReader,
    get_response_synthesizer
)
from llama_index.embeddings.cohere import CohereEmbedding
from llama_index.llms.cohere import Cohere
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.core.node_parser import MarkdownNodeParser
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core.chat_engine import ContextChatEngine # לניהול צ'אט עם זיכרון

# --- NetFree & SSL Bypass ---
os.environ['CURL_CA_BUNDLE'] = ""
os.environ['HTTPX_VERIFY'] = "False" 
ssl._create_default_https_context = ssl._create_unverified_context
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

unsafe_client = httpx.Client(verify=False) 

load_dotenv()
print("✅ Step 2: All Imports and NetFree bypass configured.")

✅ Step 2: All Imports and NetFree bypass configured.


In [3]:
load_dotenv()

cohere_key = os.getenv("COHERE_API_KEY")

if not cohere_key:
    print("Error: COHERE_API_KEY not found in .env file.")
else:
    print("✅ Step 3: API Keys loaded.")    

✅ Step 3: API Keys loaded.


In [4]:
# --- 1. Setup Cohere ---
Settings.embed_model = CohereEmbedding(
    api_key=os.getenv("COHERE_API_KEY"), 
    model_name="embed-multilingual-v3.0", 
    http_client=unsafe_client
)
Settings.llm = Cohere(
    api_key=os.getenv("COHERE_API_KEY"), 
    model="command-r-08-2024"
)

# --- 2. Setup Node Parser ---
Settings.node_parser = MarkdownNodeParser()

# --- 3. Setup Pinecone ---
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"), ssl_verify=False)
pinecone_index = pc.Index("rag-project")
vector_store = PineconeVectorStore(pinecone_index=pinecone_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# --- 4. Setup Advanced RAG ---
def setup_query_engine(index):
    retriever = VectorIndexRetriever(index=index, similarity_top_k=3)
    response_synthesizer = get_response_synthesizer(response_mode="compact")
    node_postprocessors = [SimilarityPostprocessor(similarity_cutoff=0.5)]
    
    return RetrieverQueryEngine(
        retriever=retriever,
        response_synthesizer=response_synthesizer,
        node_postprocessors=node_postprocessors
    )

print("✅ Step 4: Models, Parser, Pinecone, and RAG Pipeline configured.")

✅ Step 4: Models, Parser, Pinecone, and RAG Pipeline configured.


In [5]:
print("✅ Step 5: Loading and Chunking documents...")

project_configs = {
    "Project_1": {"path": "../Data/cloude/project1", "type": "Cloud"},
    "Project_2": {"path": "../Data/cloude/project2", "type": "Cloud"},
    "Kiro_Logic": {"path": "../Data/kiro", "type": "Kiro"}
}

all_docs = []

print("--- Starting Data Loading ---")

for proj_id, config in project_configs.items():
    path = config["path"]
    tool_type = config["type"]
    
    try:
        if os.path.exists(path):
            documents = SimpleDirectoryReader(path).load_data()
            
            for doc in documents:
                doc.metadata["project_id"] = proj_id
                doc.metadata["tool_type"] = tool_type
                doc.metadata["file_name"] = doc.metadata.get("file_name", "unknown")
            
            all_docs.extend(documents)
            print(f"✅ {proj_id} ({tool_type}): Loaded {len(documents)} docs.")
        else:
            print(f"❌ Path not found: {path}")
            
    except Exception as e:
        print(f"💥 Error in {proj_id}: {str(e)}")

print(f"\nTotal documents: {len(all_docs)}")

nodes = Settings.node_parser.get_nodes_from_documents(all_docs)

print(f"✅ Created {len(nodes)} chunks (Nodes) using Markdown Strategy.")

✅ Step 5: Loading and Chunking documents...
--- Starting Data Loading ---
✅ Project_1 (Cloud): Loaded 3 docs.
✅ Project_2 (Cloud): Loaded 3 docs.
✅ Kiro_Logic (Kiro): Loaded 3 docs.

Total documents: 9
✅ Created 213 chunks (Nodes) using Markdown Strategy.


In [ ]:
print("✅ Step 6: Starting Indexing process...")

try:
    index = VectorStoreIndex(
        nodes,
        storage_context=storage_context, 
        show_progress=True
    )
    print("✨ SUCCESS: Nodes are indexed in Pinecone!")
    
except Exception as e:
    print(f"❌ Indexing failed: {e}")

In [7]:
print("🚀 Step 7: Building the Ultimate Integrated RAG Pipeline")

retriever = index.as_retriever(similarity_top_k=12)
node_postprocessor = SimilarityPostprocessor(similarity_cutoff=0.12)
response_synthesizer = get_response_synthesizer(response_mode="tree_summarize")

chat_engine = ContextChatEngine.from_defaults(
    retriever=retriever,
    node_postprocessors=[node_postprocessor],
    response_synthesizer=response_synthesizer,
    system_prompt=(
        "You are a professional AI assistant for developers. "
        "IMPORTANT: Always respond in the same language as the user's question. "
        "Analyze the context carefully. You have information on several distinct projects: "
        "1. Project 1: Task Management application (React, Vite, SQLite). "
        "2. Project 2: Logistics and Inventory Management (Azure, SQL, Purchase Orders). "
        "3. Kiro Logic / Project 3: Medical/Clinic system (HIPAA, Security, Encryption). "
        "Do not mix their details or use the same table structure for all of them. "
        "If a project like Project 4 or 5 is mentioned, explicitly say you have no information about them. "
        "Answer accurately based ONLY on the provided context."
    )
)

def predict(message, history):
    try:
        response = chat_engine.chat(message)
        return str(response)
    
    except Exception as e:
        return f"❌ Error: {str(e)}"

print("✅ Step 7: Final Integrated Version is ready! Now run Step 9.")

🚀 Step 7: Building the Ultimate Integrated RAG Pipeline
✅ Step 7: Final Integrated Version is ready! Now run Step 9.


In [ ]:
import gradio as gr

# העיצוב המקורי שלך
custom_css = """
.gradio-container {
    max-width: 100% !important; 
    width: 100% !important;
    margin: 0 !important;
    padding: 20px !important;
    direction: rtl; 
}
#chatbot {
    height: 750px !important; 
    width: 100% !important;
    border-radius: 15px;
    box-shadow: 0 4px 25px rgba(0,0,0,0.1);
}
.message-wrap .message-text {
    text-align: right;
    direction: rtl;
}
@keyframes fadeInMsg { 
    from { opacity: 0; transform: translateY(10px); } 
    to { opacity: 1; transform: translateY(0); } 
}
.message-row { animation: fadeInMsg 0.3s ease-out; }
"""

theme = gr.themes.Soft(
    primary_hue="blue", 
    neutral_hue="zinc", 
    font=[gr.themes.GoogleFont("Assistant"), "sans-serif"] 
)

with gr.Blocks(title="AI Developer Assistant", fill_width=True) as demo:
    gr.Markdown("# 🤖 AI Agentic Coding Assistant")
    gr.Markdown("Ask me anything about your project logs.")
    gr.Markdown("---") 

    gr.ChatInterface(
        fn=predict,
        chatbot=gr.Chatbot(elem_id="chatbot", show_label=False),
        textbox=gr.Textbox(
            placeholder="Type your question here...",
            scale=7,
            show_label=False
        ),
        submit_btn="Send 🚀",
        examples=[
           "What are the main tasks in Project 1?",
            "Summarize the Kiro logic.",
            "Show me the UI decisions."
        ]
    )

if __name__ == "__main__":
    demo.launch(
        share=True, 
        theme=theme, 
        css=custom_css
    )

C:\Users\user1\AppData\Local\Temp\ipykernel_15208\3683372945.py:40: UserWarning: You provided a custom `textbox` component, but also specified `submit_btn` parameter(s) on `gr.ChatInterface`. These ChatInterface parameters will be ignored. To customize these settings, pass them directly to your `gr.Textbox` or `gr.MultimodalTextbox` component instead. For example: textbox=gr.Textbox(..., submit_btn='submit')
  gr.ChatInterface(
2026-03-25 02:02:56,060 - INFO - HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7860


2026-03-25 02:02:56,080 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"
2026-03-25 02:02:56,084 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"
2026-03-25 02:02:56,127 - INFO - HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"
2026-03-25 02:02:57,557 - INFO - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2026-03-25 02:02:57,710 - INFO - HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"
2026-03-25 02:02:58,391 - INFO - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2026-03-25 02:02:58,510 - INFO - HTTP Request: GET https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_windows_amd64.exe "HTTP/1.1 200 OK"



Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026-03-25 02:03:02,333 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-error-analytics "HTTP/1.1 200 OK"
2026-03-25 02:03:02,337 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"
2026-03-25 02:03:08,766 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-03-25 02:03:12,301 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"
